In [2]:
from dotenv import load_dotenv
from openai import OpenAI
import os
import requests
from minsearch import Index
load_dotenv()

True

In [3]:
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [4]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-09-24 14:48:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-09-24 14:48:06 (10.7 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]

--2026-09-24 14:48:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 20

In [ ]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gemini-1.5-flash",
       input=prompt,
    )
    return response.choices[0].message.content

In [5]:
def rag (question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [8]:

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [15]:
def search(question,course="llm-zoomcamp"):
    boost_dict={"question": 2.0,'section':0.5}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict={'course': course},
        num_results=5)
question="how to use the llm-zoomcamp course?"
index.search(
    question,
    boost_dict={"question": 2.0,'section':0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5)

[{'id': '20c5a1347e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?',
  'answer': 'Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIt contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.'},
 {'id': '930286278d',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Where can I find previous LLM Zoomcamp projects?',
  'answer': 'You can browse previous LLM Zoomcamp project submissions here:\n\n- [2024 projects](https://courses.datatalks.club/llm-zoomcamp-2024/projects)\n- [2025 projects](https://courses.datatalks.club/llm-zoomcamp-2025/projects)\n\nThese pages show submitted repositories and can help you understand the expected scope and quality of capstone projects.'},
 {'id': 'cce328db64',
  '

In [16]:
question="how to use the llm-zoomcamp course?"
search_results = search(question)

In [ ]:
def build_prompt(question, search_results):